# Revenue and Profit margin


In [5]:
api_key = "d651smhr01qqbln5i9s0d651smhr01qqbln5i9sg"
import requests

API_KEY = api_key
BASE_URL = "https://finnhub.io/api/v1"


def get_annual_revenue_and_margin(symbol):
    url = f"{BASE_URL}/stock/financials-reported"
    params = {
        "symbol": symbol,
        "token": API_KEY
    }

    response = requests.get(url, params=params)
    response.raise_for_status()
    data = response.json()

    results = []

    for item in data.get("data", []):
        report = item.get("report", {})
        period = report.get("period")

        # Only annual reports
        if period != "annual":
            continue

        fin = report.get("fin", {})
        revenue = fin.get("Revenue", {}).get("value")
        net_income = fin.get("NetIncomeLoss", {}).get("value")

        if revenue and net_income is not None:
            profit_margin = net_income / revenue
        else:
            profit_margin = None

        results.append({
            "fiscal_year": report.get("fiscalYear"),
            "revenue": revenue,
            "net_income": net_income,
            "profit_margin": profit_margin
        })

    return results


symbol = "AAPL"  # any public company ticker
financials = get_annual_revenue_and_margin(symbol)
print(f"Financials for {financials}:")
for f in financials:
    print(
        f"Year: {f['fiscal_year']} | "
        f"Revenue: {f['revenue']} | "
        f"Profit Margin: "
        f"{round(f['profit_margin'] * 100, 2) if f['profit_margin'] else 'N/A'}%"
    )


Financials for []:


In [9]:
import re

def extract_company_name_from_bse_text(text: str) -> str | None:
    # Find first 6-digit BSE scrip code
    m = re.search(r"\b\d{6}\b", text)
    if not m:
        return None

    # Take everything before the scrip code, then keep only the last non-empty line
    pre = text[:m.start()]
    lines = [ln.strip() for ln in pre.splitlines() if ln.strip()]
    candidate = lines[-1] if lines else pre.strip()

    # Clean trailing separators / currency artifacts seen in some titles
    candidate = re.sub(r"[-\s]+$", "", candidate)          # trailing "-" or spaces
    candidate = re.sub(r"[-\s]*[₹$]+$", "", candidate)     # trailing "$" or "₹" (and "-$")
    candidate = re.sub(r"[-\s]+$", "", candidate).strip()  # once more, then trim

    return candidate or None


tests = [
"13 febv 26 \n Desco Infratech Ltd - 544387 - Announcement under Regulation 30 (LODR)-Award_of_Order_Receipt_of_Order",
"13 febv 26- Gabion Technologies India Ltd - 544675 - Announcement under Regulation 30 (LODR)-Award_of_Order_Receipt_of_Order",
"HFCL Ltd - 500183 - Announcement under Regulation 30 (LODR)-Award_of_Order_Receipt_of_Order",
"Highway Infrastructure Ltd - 544477 - Announcement under Regulation 30 (LODR)-Award_of_Order_Receipt_of_Order",
"Seamec Ltd - 526807 - Announcement under Regulation 30 (LODR)-Award_of_Order_Receipt_of_Order",
"17 Feb 2026\nHazoor Multi Projects Ltd-$ - 532467 - Announcement under Regulation 30 (LODR)-Award_of_Order_Receipt_of_Order ..."
]

for t in tests:
    print(extract_company_name_from_bse_text(t))

Desco Infratech Ltd
13 febv 26- Gabion Technologies India Ltd
HFCL Ltd
Highway Infrastructure Ltd
Seamec Ltd
Hazoor Multi Projects Ltd


In [12]:
import re
import pandas as pd
import os

BSE_LIST_URL = os.path.join(os.getcwd(), 'BSE_data', 'List_of_companies.csv')

def extract_scrip_code(text: str) -> str | None:
    m = re.search(r"\b\d{6}\b", text)
    return m.group(0) if m else None

def load_bse_company_map() -> dict[str, str]:
    df = pd.read_csv(BSE_LIST_URL)
    # Columns include: "Scrip code", "Security Name", "ISIN", ...
    df["Scrip code"] = df["Scrip code"].astype(str).str.zfill(6)
    return dict(zip(df["Scrip code"], df["Security Name"]))

def canonical_bse_name_from_text(text: str, code_to_name: dict[str, str]) -> str | None:
    code = extract_scrip_code(text)
    if not code:
        return None
    return code_to_name.get(code)

# usage:
code_to_name = load_bse_company_map()
print(canonical_bse_name_from_text("542377", code_to_name))

Aar Shyam India Investment Company Ltd
